# Module 2: Error analysis

1. See how a trace is made.
2. **Exercise 2 (open coding):** read traces and write a short note on anything wrong.
3. **Exercise 3 (axial coding):** group your notes into failure modes and count them.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q litellm
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. How traces are made

We start from a set of test questions, written to cover different topics and different kinds of users.

In [ ]:
questions = pd.read_csv("data/test_questions.csv")
questions.head(10)

Each question goes through the bot, and we save the **whole trace**: everything the model saw
(the system prompt and the retrieved chunks) plus its answer. Here's what the model sees for one question:

In [ ]:
from beefcake.bot import build_messages
from beefcake.retrieval import retrieve

q = "it keeps disconnecting"
for message in build_messages(q, retrieve(q)):
    print(message["role"].upper() + ":")
    print(message["content"], "\n")

`scripts/generate_traces.py` runs every test question through the bot and writes one row per trace to a CSV.
The traces you'll read today are pre-generated, so nobody waits on API calls and everyone sees the same
failures. If you have a key and want your own, set `RUN_LIVE = True`.

In [ ]:
RUN_LIVE = False  # optional: generate your own traces with a live model (a minute or two)

if RUN_LIVE and llm.has_api_key():
    from beefcake.bot import answer
    from beefcake.traces import save_traces
    traces = [answer(r["User Query"], query_topic=r["Query Topic"], trace_id=r["Question ID"])
              for _, r in questions.iterrows()]
    save_traces(traces, "data/my_traces_v1.0.csv")
    print("Saved data/my_traces_v1.0.csv")

## 2. Exercise 2: Open coding (20 min)

**On your own first (12 min).** Read each trace, including the history, and write either `PASS` or a short
note about what's wrong. Put on your product owner hat: the bot should be helpful, stick to our docs and
policies, and never make up a policy. Don't fix anything yet. Just write it down.

**Then compare with your partner (8 min).** Notice where you disagree.

You can work in a spreadsheet instead: open `data/exercise2_traces.csv` in Google Sheets or Excel and use
the "Your open code" column. The docs are in the `docs/` folder if you want to check a policy.

In [ ]:
from beefcake.traces import load_traces, show_trace

ex2 = load_traces("exercise2_traces.csv")
print(len(ex2), "traces")

In [ ]:
i = 0  # change this number and rerun to move through the traces
show_trace(ex2, i)

In [ ]:
# Your open codes: "PASS" or a short note about what's wrong.
open_codes = {
    # "T01": "Assumed 'it' was the Row",
    # "T04": "PASS",
}

In [ ]:
ex2["Your open code"] = ex2["Trace ID"].map(open_codes).fillna("")
ex2.to_csv("data/my_open_codes.csv", index=False)
print(f"Saved {(ex2['Your open code'] != '').sum()} open codes to data/my_open_codes.csv")

## 3. Exercise 3: Axial coding (12 min)

In groups of four, put all your open codes together. Group the ones that describe the same problem,
give each group a short name, and count the traces in each. Aim for categories you could actually fix:
one giant "bad answer" group is too broad, and fifteen groups of one is too narrow.

In [ ]:
# failure mode name -> the trace IDs in that group
failure_modes = {
    # "Invents policy": ["T03", "T12"],
}

counts = pd.DataFrame([{"Failure mode": name, "Traces": len(ids)} for name, ids in failure_modes.items()])
counts.sort_values("Traces", ascending=False) if len(counts) else print("Add some groups above")

### Compare with ours (after the debrief)

This is how we labeled all 25 traces.

In [ ]:
from beefcake.evals import FAILURE_MODES

ours = load_traces("traces_v1_labeled.csv")
pd.DataFrame({"Failure mode": FAILURE_MODES,
              "Traces (of 25)": [int((ours[m] == "FAIL").sum()) for m in FAILURE_MODES]})

In [ ]:
ours[["Trace ID", "User Query", "Open code"]]